# 19 · Extra place sources for fixing shared-point locations (Part D, step D3b)

**Why:** after Part B, 18,665 India schools and 13,289 health facilities still share one point with 3 or more
others (village or cluster medians, or default points; notebook 18). Their buildings can only be found from
another source that names them. This notebook collects two open sources; notebook 20 matches and validates.

**1. Overture Maps Places**, release **2026-10-21.0** (Overture Maps Foundation, `docs.overturemaps.org`).
Places of interest gathered from Meta, Microsoft, Foursquare and others; **no OpenStreetMap data**. Licence
per record: **CDLA-Permissive-2.0**, Foursquare records **Apache-2.0** (kept in `source_license`). Read with DuckDB
straight from the public files (`s3://overturemaps-us-west-2/release/2026-10-21.0/theme=places/`, ~12 GB worldwide);
only places in a box around India whose taxonomy is a school (`education>place_of_learning>school…`) or health care
(`health_care…`) are kept → `data/raw/overture/overture_places_india_2026-10-21.0.parquet`. The scan reads the
whole release once (~10–20 min).

**2. OpenStreetMap health places, all India** (Geofabrik zone extracts, ODbL 1.0, "© OpenStreetMap contributors"),
the same method as notebook 10: `amenity=hospital/clinic/doctors/health_post/dispensary` or any `healthcare=*`,
points as they are, areas by a point inside them → `data/raw/osm/india/osm_health_<zone>_<date>.json`; each zone
file is deleted after extraction. OSM schools are already in `data/raw/osm/india/` (notebook 10).

A source already saved is not downloaded again.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Imports and settings

In [3]:
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
import datetime as dt
import json
import re
import shutil
import time

import duckdb
import pyogrio
import requests
import urllib3.util.connection

urllib3.util.connection.HAS_IPV6 = False  # Geofabrik over IPv6 times out from this network (see notebook 14)
OSM_DIR = PROJECT / "data" / "raw" / "osm" / "india"
OV_DIR = PROJECT / "data" / "raw" / "overture"
TMP_DIR = (Path("/content/tmp") if IN_COLAB else PROJECT / "data" / "tmp") / "geofabrik"
URL = "https://download.geofabrik.de/asia/india/{}-zone-latest.osm.pbf"
ZONES = ["north-eastern", "eastern", "western", "northern", "central", "southern"]
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}
free_gb = shutil.disk_usage(PROJECT).free / 1e9
print(f"free disk: {free_gb:.1f} GB")
assert free_gb > 5, "less than 5 GB free: stop"

free disk: 13.6 GB


### `download()` (as notebook 10)

In [4]:
def download(zone):
    """Stream one Geofabrik zone extract to a temporary file (deleted after the schools are taken out)."""
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    f = TMP_DIR / f"{zone}-zone-latest.osm.pbf"
    if f.exists():  # only complete downloads get this name (see .part below)
        return f
    part = f.with_suffix(".part")
    with requests.get(URL.format(zone), headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(part, "wb") as out:
            shutil.copyfileobj(r.raw, out, length=1 << 20)
    part.rename(f)
    print(f"  downloaded {f.name}: {f.stat().st_size / 1e6:.0f} MB")
    return f

### `tag()` (as notebook 10)

In [5]:
def tag(other_tags, key):
    """Value of one key in GDAL's other_tags text ('"key"=>"value",...')."""
    if not isinstance(other_tags, str):  # missing tags come back as NaN
        return None
    m = re.search(r'"' + re.escape(key) + r'"=>"((?:[^"\\]|\\.)*)"', other_tags)
    return m.group(1) if m else None

## Overture Maps Places, India

In [6]:
# Overture Maps Places, India: schools and health places only (DuckDB reads only what it needs from the public files)
OV_RELEASE = "2026-10-21.0"
OV_SRC = f"s3://overturemaps-us-west-2/release/{OV_RELEASE}/theme=places/type=place/*.parquet"
INDIA_BBOX = (68.0, 6.5, 97.5, 37.2)  # lon/lat box around India (the WorldPop grid's extent)
# taxonomy paths kept: every kind of school (not coaching / driving / language schools) and all health care
CATEGORY_RE = "^(education>place_of_learning>school|health_care)"


def overture_india():
    out = OV_DIR / f"overture_places_india_{OV_RELEASE}.parquet"
    if out.exists():
        return out
    OV_DIR.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.sql("INSTALL httpfs; LOAD httpfs; SET s3_region='us-west-2';")
    x0, y0, x1, y1 = INDIA_BBOX
    t = time.time()
    con.sql(f"""COPY (
        SELECT id, names."primary" AS name, basic_category, taxonomy."primary" AS taxonomy_primary,
               array_to_string(taxonomy.hierarchy, '>') AS taxonomy_path, confidence,
               (bbox.xmin + bbox.xmax) / 2 AS lon, (bbox.ymin + bbox.ymax) / 2 AS lat,
               addresses[1].freeform AS address, addresses[1].locality AS locality, addresses[1].region AS region,
               sources[1].dataset AS source_dataset, sources[1].license AS source_license, operating_status
        FROM read_parquet('{OV_SRC}')
        WHERE bbox.xmin BETWEEN {x0} AND {x1} AND bbox.ymin BETWEEN {y0} AND {y1}
          AND regexp_matches(array_to_string(taxonomy.hierarchy, '>'), '{CATEGORY_RE}')
    ) TO '{out.as_posix()}' (FORMAT parquet, COMPRESSION zstd)""")
    print(f"Overture India places written in {(time.time() - t) / 60:.1f} min")
    return out


ov_file = overture_india()

Overture India places written in 13.1 min


## OSM health places, all India

In [7]:
HEALTH_AMENITY = ("hospital", "clinic", "doctors", "health_post", "dispensary")


def extract_health(pbf, zone):
    """amenity=hospital/clinic/doctors/... or any healthcare=* (GDAL OSM driver); areas by a point inside them."""
    s = lambda v: v if isinstance(v, str) and v else None
    like = " OR ".join([f"other_tags LIKE '%\"amenity\"=>\"{a}\"%'" for a in HEALTH_AMENITY] + ["other_tags LIKE '%\"healthcare\"=>%'"])
    els = []
    pts = pyogrio.read_dataframe(pbf, layer="points", columns=["osm_id", "name", "other_tags"], where=like)
    for i, name, other, g in zip(pts["osm_id"], pts["name"], pts["other_tags"], pts.geometry):
        els.append({"type": "node", "id": int(i), "lat": g.y, "lon": g.x,
                    "tags": {k: v for k, v in [("name", s(name)), ("name:en", tag(other, "name:en")),
                                               ("amenity", tag(other, "amenity")), ("healthcare", tag(other, "healthcare"))] if v}})
    am = ", ".join(f"'{a}'" for a in HEALTH_AMENITY)
    polys = pyogrio.read_dataframe(pbf, layer="multipolygons", columns=["osm_id", "osm_way_id", "name", "amenity", "other_tags"],
                                   where=f"amenity IN ({am}) OR other_tags LIKE '%\"healthcare\"=>%'")
    for rid, wid, name, amen, other, g in zip(polys["osm_id"], polys["osm_way_id"], polys["name"], polys["amenity"],
                                              polys["other_tags"], polys.geometry):
        p = g.representative_point()
        els.append({"type": "relation" if s(rid) else "way", "id": int(s(rid) or wid), "center": {"lat": p.y, "lon": p.x},
                    "tags": {k: v for k, v in [("name", s(name)), ("name:en", tag(other, "name:en")), ("amenity", s(amen)),
                                               ("healthcare", tag(other, "healthcare"))] if v}})
    return {"source": URL.format(zone), "generator": "GDAL OSM driver (pyogrio)", "elements": els}


def run_osm_health():
    OSM_DIR.mkdir(parents=True, exist_ok=True)
    for zone in ZONES:
        if list(OSM_DIR.glob(f"osm_health_{zone}_*.json")):
            print(zone, "already saved")
            continue
        assert shutil.disk_usage(PROJECT).free / 1e9 > 5, "less than 5 GB free: stop"
        pbf = download(zone)
        d = extract_health(pbf, zone)
        f = OSM_DIR / f"osm_health_{zone}_{dt.date.today().isoformat()}.json"
        f.write_text(json.dumps(d), encoding="utf-8")
        pbf.unlink()
        print(f"{zone}: {len(d['elements'])} health places -> {f.name}")


run_osm_health()

  downloaded north-eastern-zone-latest.osm.pbf: 110 MB


north-eastern: 4144 health places -> osm_health_north-eastern_2026-11-08.json


  downloaded eastern-zone-latest.osm.pbf: 248 MB


C:\Users\ASUS\miniforge3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(


eastern: 16359 health places -> osm_health_eastern_2026-11-08.json


  downloaded western-zone-latest.osm.pbf: 222 MB


... (5 outputs left out)

central: 19131 health places -> osm_health_central_2026-11-08.json


  downloaded southern-zone-latest.osm.pbf: 559 MB


C:\Users\ASUS\miniforge3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(


southern: 51167 health places -> osm_health_southern_2026-11-08.json


## Check

In [8]:
import pandas as pd

ov = pd.read_parquet(ov_file)
print(f"Overture {OV_RELEASE}: {len(ov):,} school/health places in the India box, {ov_file.stat().st_size / 1e6:.0f} MB")
print(ov["basic_category"].value_counts().head(25).to_string())
print(ov["source_license"].value_counts().to_string())
tot = 0
for f in sorted(OSM_DIR.glob("osm_health_*.json")):
    els = json.loads(f.read_text(encoding="utf-8"))["elements"]
    tot += len(els)
    print(f"{f.name:50s} {len(els):7d} places, {sum(1 for e in els if e['tags'].get('name')):7d} named")
print("OSM health total:", tot)

Overture 2026-10-21.0: 571,652 school/health places in the India box, 32 MB
basic_category
place_of_learning                         163804
hospital                                  105982
diagnostics_imaging_or_lab_service         54616
dental_clinic                              37681
health_care                                35136
high_school                                31174
preschool                                  24762
elementary_school                          20072
specialized_health_care                    19957
complementary_and_alternative_medicine     15309
physical_medicine_and_rehabilitation       10907
outpatient_care_facility                    9682
medical_service                             7970
reproductive_perinatal_and_womens_care      6424
vision_or_eye_care_clinic                   6216
behavioral_or_mental_health_clinic          5381
surgery                                     4782
middle_school                               3648
specialized_medical_facilit

osm_health_southern_2026-11-08.json                  51167 places,   49231 named
osm_health_western_2026-11-08.json                   25917 places,   25476 named
OSM health total: 138434
